Tweet Generator Workflow example of iterative workflow


In [22]:
import os 
from dotenv import load_dotenv
from math import sqrt
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START , END, StateGraph
from langchain_core.messages import SystemMessage, HumanMessage
from typing import TypedDict, Annotated, Literal, Union
from pydantic import BaseModel , Field
import time
load_dotenv()
# Ideally use the best models excelling in respective capactilities for different tasks, for sake of implementation we are using same models
generator_llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [23]:
class TweetState(TypedDict):
    topic : str
    tweet : str
    evaluation : Literal["approved", "needs_improvement"] 
    feedback  : str
    iteration : int
    max_iterations : int

In [24]:
class EvaluateSchema(BaseModel):
    evaluate : Literal["approved", "needs_improvement"] = Field(description="Evaluates the tweet and marks as approved or needs_improvement")
    feedback : str = Field(description="feedback for the tweet")
evaluator_llm= model.with_structured_output(EvaluateSchema)

In [27]:
def generate_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever Twitter's influencer"), 
        HumanMessage(content=f"""
Write a short, original , hilarious tweet on the topic: "{state['topic']}". 
Rules:
Don't use question answer format. 
Max 280 characters
Use observational humor, irony , sarcasm , or cultural references.
Think in meme logic, punchlines or relatable lines.
""")
    ]
    response = generator_llm.invoke(messages).content     
    return {"tweet"  : response}
def evaluate_tweet(state : TweetState):
    messages = [SystemMessage(content =" You are a ruthless , no laugh given ruthless Twitter critic. You evaluate tweets based on humor, originality, virality,and tweet format "), 
                HumanMessage(content=f"""Evaluate the following tweet. Tweet:  "{state["tweet"]}"
                        
You are a social media expert tasked with reviewing tweets before posting. Evaluate the following tweet based on these criteria:
Evaluation Criteria:
Originality: Does the tweet present a fresh or unique idea?
Humor: Is the tweet witty or funny?
Punchiness: Is the tweet impactful and concise?
Virality
Tweet Format:
The tweet should not be in question/answer format.
It should be structured as a well-formed single tweet.
Length: The tweet must not exceed 280 characters.
Generality: The tweet should not be too generic (e.g., "work hard and believe in yourself").
Auto-Rejection Conditions:
If any of the following is true, mark the tweet as "needs_improvement" immediately and explain why:
The tweet is over 280 characters.
It is in question/answer format.
It is too generic or lacks substance.
Respond only in structured format:
evaluate : "approved" or "needs_improvement"
feedback : one paragraph explaining strengths and weaknesses
""")]
    response = evaluator_llm.invoke(messages)
    print(response)
    return { "evaluation" : response.evaluate, "feedback" : response.feedback}
def optimize_tweet(state :TweetState):
    messages = [SystemMessage(content="""You are a social media content strategist. Your job is to take a tweet and improve it to make it more human, unique, and viral — while staying under the 280-character limit. You are not allowed to write tweets in Q&A format. Focus on improving relatability, humor, clarity, and emotional impact"""), 
    HumanMessage(content=f"""Original Tweet:
{state["tweet"]}
Feedback:
{state["feedback"]}
Please rewrite and improve the tweet based on the following points:
Human: Make it feel like it was written by a real, witty person — not robotic.
Virality: Boost its chances of being shared widely by adding emotion, humor, or surprise.
Uniqueness: Avoid clichés. Make the tweet stand out from others on the timeline.
Concise: Keep the tweet within the 280-character limit.
Format: Avoid question-and-answer style tweets.
Your output should be a single improved tweet. Do not include any explanation or extra text.""")]
    response = optimizer_llm.invoke(messages).content
    iteration = state["iteration"] + 1
    return {"tweet"  :response, "iteration" : iteration}


In [28]:
def router(state :TweetState):
    if state["evaluation"] == "approved" or state["iteration"] >= state["max_iterations"] :
        return "approved"
    return "needs_improvement"
graph = StateGraph(TweetState)

graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize_tweet", optimize_tweet)

graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "evaluate")
graph.add_conditional_edges("evaluate", router, {'approved': END, 'needs_improvement' : 'optimize_tweet'})
graph.add_edge("optimize_tweet", "evaluate")
workflow = graph.compile()

In [29]:
initial_state = {
    "topic" : "Pakistani Railways", 
    "iteration" : 0, 
    "max_iterations" : 5
}
result = workflow.invoke(initial_state)
print(result)

evaluate='approved' feedback="This tweet is pretty solid. The naan aging analogy is original and funny. The beard line is a great punchline. Plus, the hashtags are relevant. It's a well-structured tweet that fits the character limit, so it should resonate well."
{'topic': 'Pakistani Railways', 'tweet': 'My Pakistani Railways experience was so authentic, I think I aged like a fine naan. By the time I reached my destination, my beard was longer than the train itself. #Pakistan #Travel #MaybeWalkingNextTime', 'evaluation': 'approved', 'feedback': "This tweet is pretty solid. The naan aging analogy is original and funny. The beard line is a great punchline. Plus, the hashtags are relevant. It's a well-structured tweet that fits the character limit, so it should resonate well.", 'iteration': 0, 'max_iterations': 5}
